# Día 4 · Unidad 4 — Funciones (II): `*args` y `**kwargs`

**Objetivos de aprendizaje**
- Pasar un número variable de argumentos posicionales con `*args`.
- Pasar un número variable de argumentos `keyword` con `**kwargs`.
- Ordenar correctamente parámetros estándar, `*args` y `**kwargs` en una misma definición.
- Leer e interpretar un `SyntaxError` de Python cuando el orden es incorrecto.
- Usar los operadores de *unpacking* `*` y `**` sobre listas y diccionarios.

**Duración estimada:** 60 minutos.

**Requisitos previos:** `01_teoria_funciones_basicas.ipynb` de este mismo día.

## 1. El problema: un número variable de argumentos

Queremos sumar el importe de varios siniestros:

In [1]:
def sumar_siniestros(a, b):
    return a + b


sumar_siniestros(120.0, 340.0)


460.0

¿Y si algún mes hay tres siniestros? ¿O cinco? Una opción poco elegante es poner muchos parámetros con valor por defecto:

In [2]:
def sumar_siniestros(a=0, b=0, c=0, d=0, e=0):
    return a + b + c + d + e


sumar_siniestros(120.0, 340.0, 90.0)


550.0

Funciona, pero tiene un límite fijo (cinco) y arrastra parámetros vacíos casi siempre. Otra opción es aceptar una lista:

In [3]:
def sumar_siniestros(importes):
    total = 0
    for importe in importes:
        total += importe
    return total


sumar_siniestros([120.0, 340.0, 90.0, 75.0])


625.0

Esto sí es flexible, pero obliga a quien llama a construir siempre una lista, incluso para un único importe. Aquí es donde entra `*args`.

## 2. `*args`: número variable de argumentos posicionales

Con `*args` en la definición, la función acepta cualquier cantidad de argumentos posicionales, y los recibe agrupados en una **tupla**:

In [4]:
def sumar_siniestros(*args):
    total = 0
    for importe in args:
        total += importe
    return total


sumar_siniestros(120.0, 340.0, 90.0, 75.0, 60.0)


685.0

> **Nota**: el nombre `args` es sólo una convención, no una palabra reservada — podrías llamarlo `*importes` y funcionaría igual. Pero usa `*args` salvo que tengas una razón de peso: es lo que cualquier programador de Python reconoce al instante.

In [5]:
def sumar_siniestros(*importes):
    return sum(importes)


sumar_siniestros(120.0, 340.0, 90.0)


550.0

> **Nota**: dentro de la función, `args` (o el nombre que le hayas puesto) es una **tupla**, no una lista:

In [6]:
def inspeccionar_args(*args):
    print(type(args))
    print(args)


inspeccionar_args(120.0, 340.0, 90.0)


<class 'tuple'>
(120.0, 340.0, 90.0)


## 3. `**kwargs`: número variable de argumentos `keyword`

`**kwargs` funciona de forma parecida, pero para argumentos del tipo `keyword=valor`. Dentro de la función, `kwargs` es un **diccionario**.

In [7]:
def crear_poliza(**kwargs):
    for campo, valor in kwargs.items():
        print(f"{campo}: {valor}")


crear_poliza(numero="POL-2024-0451", producto="auto", prima=420.0)


numero: POL-2024-0451
producto: auto
prima: 420.0


In [8]:
def inspeccionar_kwargs(**kwargs):
    print(type(kwargs))


inspeccionar_kwargs(numero="POL-2024-0451", producto="auto")


<class 'dict'>


**Para ti:** escribe una función `resumen_cartera(*primas)` que use `*args` para devolver una tupla `(numero_polizas, prima_total, prima_media)` a partir de un número variable de primas. Pruébala con 3 primas y con 6.

## 4. Orden de los parámetros

Si una función necesita parámetros estándar, `*args` **y** `**kwargs` a la vez, el orden importa: primero los parámetros estándar, después `*args`, después `**kwargs`.

In [9]:
def registrar_poliza(numero_poliza, producto, *coberturas_adicionales, **datos_cliente):
    print(f"Póliza {numero_poliza} ({producto})")
    print("Coberturas adicionales:", coberturas_adicionales)
    print("Datos del cliente:", datos_cliente)


registrar_poliza("POL-001", "auto", "lunas", "asistencia", nombre="Marta", provincia="Barcelona")


Póliza POL-001 (auto)
Coberturas adicionales: ('lunas', 'asistencia')
Datos del cliente: {'nombre': 'Marta', 'provincia': 'Barcelona'}


> ⚠️ **Error típico**: escribir `**kwargs` antes que `*args` en la definición. Es un error de sintaxis — Python ni siquiera consigue interpretar la definición, así que el fallo ocurre **al leer el código**, no al ejecutarlo.

Un `SyntaxError` normal, si simplemente escribieras esa celda de código en un notebook, detendría la ejecución del propio notebook — no se puede "capturar" con un `try/except` como una excepción normal, porque ocurre al analizar el código (parsear), antes de que llegue a ejecutarse ninguna línea.

Para poder ver el error de forma segura, en vez de escribir el código inválido directamente, se lo pasamos como texto a `compile()`. `compile()` sí intenta analizar ese texto como código Python, y si falla, lanza el `SyntaxError` como una excepción normal que **sí** podemos capturar:

In [10]:
codigo_invalido = "def func(**kwargs, *args): pass"

try:
    compile(codigo_invalido, "<string>", "exec")
except SyntaxError as e:
    print(f"{type(e).__name__}: {e}")


SyntaxError: arguments cannot follow var-keyword argument (<string>, line 1)


El mensaje confirma justo lo que dice la norma: `*args` debe declararse antes que `**kwargs`. El orden correcto es siempre:

1. Argumentos estándar (posicionales o con valor por defecto)
2. `*args`
3. `**kwargs`

```python
def func(a, b, *args, **kwargs):   # correcto
    ...

def func(a, b=0, *args, **kwargs):  # correcto
    ...
```

## 5. Operadores de *unpacking*: `*` y `**`

Además de en la definición de una función, `*` y `**` se usan para **desempaquetar** un iterable o un diccionario. `*` funciona sobre cualquier iterable; `**` sólo sobre diccionarios.

In [11]:
coberturas = ["lunas", "asistencia", "robo"]
print(coberturas)
print(*coberturas)


['lunas', 'asistencia', 'robo']
lunas asistencia robo


In [12]:
primas_trimestrales = [310.0, 295.0, 340.0, 355.0]

primer_trimestre, *resto, ultimo_trimestre = primas_trimestrales

print(primer_trimestre)
print(resto)
print(ultimo_trimestre)


310.0
[295.0, 340.0]
355.0


También sirve para combinar listas sin usar `.extend()`, o para llamar a una función pasándole los elementos de una lista como argumentos posicionales:

In [13]:
cartera_auto = ["POL-001", "POL-002"]
cartera_hogar = ["POL-101", "POL-102", "POL-103"]

cartera_completa = [*cartera_auto, *cartera_hogar]
print(cartera_completa)

print(sumar_siniestros(*[120.0, 340.0, 90.0]))


['POL-001', 'POL-002', 'POL-101', 'POL-102', 'POL-103']
550.0


Con `**` se combinan diccionarios — muy útil para juntar una configuración base con overrides puntuales, por ejemplo al construir los parámetros de una póliza a partir de una plantilla del producto:

In [14]:
parametros_base_auto = {"producto": "auto", "franquicia": 300.0, "cobertura": "terceros"}
overrides_cliente = {"cobertura": "todo riesgo", "conductor_novel": True}

parametros_poliza = {**parametros_base_auto, **overrides_cliente}
print(parametros_poliza)


{'producto': 'auto', 'franquicia': 300.0, 'cobertura': 'todo riesgo', 'conductor_novel': True}


Fíjate en `"cobertura"`: aparece en los dos diccionarios, y al combinarlos con `**` gana el valor del **segundo** (`overrides_cliente`) — el mismo comportamiento que si asignaras las claves una por una, en ese orden.

**Para ti:** dada `plantilla_hogar = {"producto": "hogar", "franquicia": 150.0}` y `datos_cliente = {"nombre": "Marta", "franquicia": 100.0}`, combínalos con `**` en ese orden y comprueba qué valor de `franquicia` prevalece. Luego invierte el orden y comprueba que cambia.

## Resumen

- `*args` agrupa un número variable de argumentos posicionales en una tupla dentro de la función.
- `**kwargs` agrupa un número variable de argumentos `keyword` en un diccionario dentro de la función.
- Los nombres `args`/`kwargs` son sólo convención — lo importante son los operadores `*`/`**`.
- El orden de una definición con las tres cosas es siempre: argumentos estándar, `*args`, `**kwargs`. Escribir `**kwargs` antes que `*args` es un `SyntaxError`, que ocurre al leer el código, no al ejecutarlo — por eso hace falta `compile()` dentro de un `try/except` para poder capturarlo de forma controlada en un notebook.
- Los mismos operadores `*` y `**`, fuera de una definición de función, desempaquetan iterables y diccionarios: para combinarlos, para pasarlos como argumentos, o para "spread" una lista dentro de otra.

**Siguiente:** `03_teoria_scope_namespaces.ipynb`, para entender dónde "vive" cada variable y por qué a veces una función no ve la variable que esperas.